In [15]:
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

DATA_PATH = r"C:/Files/GLAN_DATA/first_wave/individual_level/integrated_dataset.csv"

col_Y = 'who_5'                 #
col_T_Green = 'proportion_greenery' # proportion_greenery, greenery_visibility
col_T_Noise = 'LAeq_overall'        #
col_M1 = 'sleep_duration'       #
col_M2 = 'MVPA_10min'           # 

cols_W = [
    "age", "gender", "education_level", "family_income",
    "PM2_5","temperature", "humidity"
]

df_raw = pd.read_csv(DATA_PATH)

all_cols = [col_Y, col_T_Green, col_T_Noise, col_M1, col_M2] + cols_W
df = df_raw[all_cols].copy()
scaler = StandardScaler()
df[all_cols] = scaler.fit_transform(df[all_cols])

print(f" {len(df)}  {len(all_cols)} ")

 658  12 


In [16]:
import pandas as pd
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tensordict import TensorDict
from causica.datasets.variable_types import VariableTypeEnum
from causica.datasets.causica_dataset_format import Variable
from causica.distributions import ContinuousNoiseDist
from causica.lightning.modules.deci_module import DECIModule
import time
import warnings
warnings.filterwarnings('ignore')

In [18]:
print("=" * 70)
print("Scenario 2: Greenery <= 5% -> 30% (low-MVPA subgroup)")
print("=" * 70)
# ============================================================================
# Load model
# ============================================================================
 
MODEL_WEIGHTS_PATH =  "deci_model_weights_LAeq_overall_greenpro.pth"#"deci_model_weights_LAeq_overall_GV.pth"
 
loaded_deci_model = DECIModule(
    noise_dist=ContinuousNoiseDist.GAUSSIAN,
    embedding_size=32,
    constraint_matrix_path="structural_constraints.npy",
)
loaded_deci_model.load_state_dict(
    torch.load(MODEL_WEIGHTS_PATH, map_location=torch.device('cpu'))
)
loaded_deci_model = loaded_deci_model.cuda()
loaded_deci_model.eval()
 
device = next(loaded_deci_model.parameters()).device
print(f"  Model loaded on {device}")
 
# ============================================================================
# Configuration
# ============================================================================
 
NUM_SEM = 20
N_MC = 1000
N_MC_NDE = 100
N_INNER = 20
MAX_INDIVIDUALS = 25
 
S2_GREEN_MAX = 5      # proportion <= 10%
S2_MVPA_MAX = 10.0       # MVPA <= 10 min
S2_GREEN_TARGET = 30   # target: 30%

# ============================================================================
# Helper functions
# ============================================================================
 
def make_interv(var_name, value):
    return TensorDict({var_name: torch.tensor([float(value)], device=device)},
                      batch_size=[])
 
def make_joint_interv(var_dict):
    return TensorDict({k: torch.tensor([float(v)], device=device)
                       for k, v in var_dict.items()},
                      batch_size=[])
 
def raw_to_std(val_raw, var_name):
    idx = all_cols.index(var_name)
    return (val_raw - scaler.mean_[idx]) / scaler.scale_[idx]
 
def effect_to_raw(eff_std, outcome_var):
    idx = all_cols.index(outcome_var)
    return eff_std * scaler.scale_[idx]
 
# ============================================================================
# Sample SEMs
# ============================================================================
 
with torch.no_grad():
    sems = list(loaded_deci_model.sem_module().sample(torch.Size([NUM_SEM])))
print(f"  SEM x {NUM_SEM}, MC x {N_MC}, MC_NDE x {N_MC_NDE}, Inner x {N_INNER}")
 
# ============================================================================
# Identify subgroup
# ============================================================================
 
mask_s2 = (df_raw[col_T_Green] <= S2_GREEN_MAX) & (df_raw[col_M2] <= S2_MVPA_MAX)
subgroup_s2 = df_raw[mask_s2]
 
print(f"\n  Subgroup: Greenery <= {S2_GREEN_MAX:.4f}% AND MVPA <= {S2_MVPA_MAX:.4f} min")
print(f"    Size: {len(subgroup_s2)}")
print(f"    Mean greenery: {subgroup_s2[col_T_Green].mean():.3f}")
print(f"    Mean MVPA:     {subgroup_s2[col_M2].mean():.1f} min")
print(f"    Mean WHO-5:    {subgroup_s2[col_Y].mean():.2f}")
 
if len(subgroup_s2) > MAX_INDIVIDUALS:
    print(f"  Downsampling {len(subgroup_s2)} -> {MAX_INDIVIDUALS}")
    subgroup_s2_use = subgroup_s2.sample(n=MAX_INDIVIDUALS, random_state=42)
else:
    subgroup_s2_use = subgroup_s2
 
# ============================================================================
# Compute mediation
# ============================================================================
 
def compute_subgroup_mediation(sems, subgroup_df, treat_var, med_var, outcome_var,
                                t_new_value_raw, n_mc, n_mc_nde, n_inner):
    t_new_std = raw_to_std(t_new_value_raw, treat_var)
    t_orig_list = [raw_to_std(row[treat_var], treat_var)
                   for _, row in subgroup_df.iterrows()]
 
    te_all, nde_all, nie_all = [], [], []
 
    for sem_idx, sem in enumerate(sems):
        with torch.no_grad():
            y_orig_list, y_new_list = [], []
            y_nde_new_list, y_nde_orig_list = [], []
 
            for t_orig in t_orig_list:
                s_orig = sem.do(make_interv(treat_var, t_orig)).sample(torch.Size([n_mc]))
                s_new = sem.do(make_interv(treat_var, t_new_std)).sample(torch.Size([n_mc]))
 
                y_orig_list.append(s_orig[outcome_var].mean().item())
                y_new_list.append(s_new[outcome_var].mean().item())
 
                m_natural = s_orig[med_var].squeeze(-1)[:n_mc_nde]
 
                y_nde_new_i, y_nde_orig_i = [], []
                for i in range(n_mc_nde):
                    m_v = m_natural[i].item()
 
                    sy_new = sem.do(make_joint_interv({treat_var: t_new_std,
                                                        med_var: m_v})) \
                                .sample(torch.Size([n_inner]))
                    y_nde_new_i.append(sy_new[outcome_var].mean().item())
 
                    sy_orig = sem.do(make_joint_interv({treat_var: t_orig,
                                                         med_var: m_v})) \
                                 .sample(torch.Size([n_inner]))
                    y_nde_orig_i.append(sy_orig[outcome_var].mean().item())
 
                y_nde_new_list.append(np.mean(y_nde_new_i))
                y_nde_orig_list.append(np.mean(y_nde_orig_i))
 
            te = np.mean(y_new_list) - np.mean(y_orig_list)
            nde = np.mean(y_nde_new_list) - np.mean(y_nde_orig_list)
            nie = te - nde
 
        te_all.append(te)
        nde_all.append(nde)
        nie_all.append(nie)
 
        if (sem_idx + 1) % 5 == 0:
            print(f"      SEM {sem_idx+1}/{len(sems)}")
 
    return np.array(te_all), np.array(nde_all), np.array(nie_all)
 
print("\nComputing...")
t_start = time.time()
te_s2, nde_s2, nie_s2 = compute_subgroup_mediation(
    sems, subgroup_s2_use,
    col_T_Green, col_M2, col_Y,
    S2_GREEN_TARGET,
    N_MC, N_MC_NDE, N_INNER
)
elapsed = time.time() - t_start
print(f"  Completed in {elapsed/60:.1f} min")
 
# Summarize
def summarize(arr, outcome_var):
    return (effect_to_raw(np.mean(arr), outcome_var),
            effect_to_raw(np.percentile(arr, 5), outcome_var),
            effect_to_raw(np.percentile(arr, 95), outcome_var))
 
te_m, te_l, te_h = summarize(te_s2, col_Y)
nde_m, nde_l, nde_h = summarize(nde_s2, col_Y)
nie_m, nie_l, nie_h = summarize(nie_s2, col_Y)
pct = (nie_m / te_m * 100) if abs(te_m) > 1e-6 else np.nan
 
print(f"\n  Results (n={len(subgroup_s2_use)}, 90% CI):")
print(f"    TE  = {te_m:>7.3f}  [{te_l:.3f}, {te_h:.3f}]")
print(f"    NDE = {nde_m:>7.3f}  [{nde_l:.3f}, {nde_h:.3f}]")
print(f"    NIE = {nie_m:>7.3f}  [{nie_l:.3f}, {nie_h:.3f}]")
print(f"    % mediated by MVPA: {pct:.1f}%")
 
# Save results
results_df = pd.DataFrame([{
    'scenario': f'S2: Green<={S2_GREEN_MAX*100:.0f}%->{S2_GREEN_TARGET*100:.0f}%, MVPA<={S2_MVPA_MAX:.0f}min',
    'mediator': 'MVPA',
    'subgroup_size_total': len(subgroup_s2),
    'subgroup_size_used': len(subgroup_s2_use),
    'TE_mean': te_m, 'TE_CI90_low': te_l, 'TE_CI90_high': te_h,
    'NDE_mean': nde_m, 'NDE_CI90_low': nde_l, 'NDE_CI90_high': nde_h,
    'NIE_mean': nie_m, 'NIE_CI90_low': nie_l, 'NIE_CI90_high': nie_h,
    'pct_mediated': pct,
}])
results_df.to_csv("scenario2_results.csv", index=False)
print("  Saved: scenario2_results.csv")
print("Done.")

Scenario 2: Greenery <= 5% -> 30% (low-MVPA subgroup)
  Model loaded on cuda:0
  SEM x 20, MC x 1000, MC_NDE x 100, Inner x 20

  Subgroup: Greenery <= 5.0000% AND MVPA <= 10.0000 min
    Size: 24
    Mean greenery: 3.552
    Mean MVPA:     5.5 min
    Mean WHO-5:    12.00

Computing...
      SEM 5/20
      SEM 10/20
      SEM 15/20
      SEM 20/20
  Completed in 146.3 min

  Results (n=24, 90% CI):
    TE  =   1.977  [1.777, 2.140]
    NDE =   1.970  [1.781, 2.074]
    NIE =   0.008  [-0.064, 0.093]
    % mediated by MVPA: 0.4%
  Saved: scenario2_results.csv
Done.
